In [1]:
from torch.utils.data import DataLoader
import torch
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup,AutoTokenizer,DistilBertModel
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import numpy as np
from torchvision.models import resnet18,ResNet18_Weights
from torchvision import transforms
from modules import CollateFunction,creation_dataframe,CreationDataset,Train,DistilbertResnetModel

In [2]:
#Creation of the dataframes from the jsonl files
train_df=creation_dataframe("../data/train.jsonl")
val_df=creation_dataframe("../data/dev.jsonl")

In [3]:
#Creation of the datasets
train_dataset=CreationDataset(train_df)
val_dataset=CreationDataset(val_df)

In [4]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [5]:
collate_object=CollateFunction(tokenizer)

In [ ]:
#Creation of the dataloaders
batch_size=32
train_dataloader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_object.collate_fn,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_object.collate_fn,drop_last=True)

In [7]:
#Use of the resnet18 model initialized with its default pretrained weights as the Vision model
resnet_model=resnet18(weights=ResNet18_Weights.DEFAULT)

In [8]:
#Use of the pretrained distilbert model as the transformer model
distilbert_model=DistilBertModel.from_pretrained("distilbert-base-uncased")

In [9]:
#Finally, use of the custom model
model=DistilbertResnetModel(distilbert_model,resnet_model)

In [10]:
#Use of the class weights to compensate imabalances of the dataset and make more accurate predictions
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight,dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [11]:
#Training hyperparameters
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
n_warmup_steps=int(0.1*n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [12]:
trainer=Train(model=model,loss_fn=loss_fn,n_epochs=n_epochs,device=device,n_steps=n_steps,n_warmup_steps=n_warmup_steps,n_frozen_distilbert_layers=6,n_frozen_resnet_layers=4)

In [13]:
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings")

2026-03-11 11:19:56.497 | INFO     | modules.train:run_training:165 - Epoch 0 :
2026-03-11 11:49:16.836 | INFO     | modules.train:run_training:230 - Epoch 0: Train Loss = 0.7298949963616249
2026-03-11 11:49:16.839 | INFO     | modules.train:run_training:231 - Epoch 0: Train Accuracy = 0.44
2026-03-11 11:49:16.840 | INFO     | modules.train:run_training:232 - Epoch 0: Train F1 = 0.41856731581282836
2026-03-11 11:49:16.841 | INFO     | modules.train:run_training:234 - Epoch 0: Validation Loss = 0.6672737970948219
2026-03-11 11:49:16.843 | INFO     | modules.train:run_training:235 - Epoch 0: Validation Accuracy = 0.518
2026-03-11 11:49:16.845 | INFO     | modules.train:run_training:236 - Epoch 0: Validation F1 = 0.4434462914124455
2026-03-11 11:49:18.296 | INFO     | modules.train:run_training:165 - Epoch 1 :


KeyboardInterrupt: 

In [ ]:
#Load of the training performances per epoch (losses, f1 scores, accuracies)
training_performances=torch.load("modules/train_savings/epoch_performances.pt")

In [15]:
print(training_performances)

{'epoch_train_losses': tensor([0.7054, 0.6488, 0.5994, 0.5595, 0.5284], dtype=torch.float64), 'epoch_train_f1': tensor([0.5482, 0.6274, 0.6928, 0.7159, 0.7369], dtype=torch.float64), 'epoch_train_accuracies': tensor([0.5414, 0.6204, 0.6885, 0.7116, 0.7329], dtype=torch.float64), 'epoch_val_losses': tensor([0.7165, 0.7755, 0.6963, 0.7693, 0.7854], dtype=torch.float64), 'epoch_val_f1': tensor([0.4824, 0.5448, 0.6178, 0.5848, 0.6010], dtype=torch.float64), 'epoch_val_accuracies': tensor([0.4920, 0.5780, 0.6200, 0.5980, 0.6080], dtype=torch.float64)}
